# Notebook 01 — Data Extraction

**Objective:** Load the raw dataset, validate its structure, and document the extraction metadata before any transformation is applied.

In [ ]:
import pandas as pd
import numpy as np
import os

from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(
    os.environ.get(
        'PROJECT_ROOT',
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    )
).resolve()

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

RAW_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv',
    PROJECT_ROOT / 'investments_VC.csv',
    Path('/content/investments_VC.csv'),
)

CLEAN_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv',
    PROJECT_ROOT / 'startups_cleaned.csv',
    Path('/content/startups_cleaned.csv'),
)

SCREENSHOT_DIR = PROJECT_ROOT / 'tableau' / 'screenshots'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

def maybe_download(path: Path) -> None:
    if IN_COLAB:
        _colab_files.download(str(path))

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Raw data path resolved to: {RAW_DATA_PATH}')

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH, encoding='latin-1', low_memory=False)

print('Dataset loaded successfully.')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

In [ ]:
df_raw.head()

In [ ]:
print('Column Names and Data Types:')
print('=' * 50)
for col in df_raw.columns:
    print(f'  {col.strip():<35} {str(df_raw[col].dtype)}')

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('Missing Value Report:')
print(missing_report[missing_report['Missing Count'] > 0].to_string())

In [ ]:
categorical_cols = ['status', 'country_code', 'state_code', 'market']

for col in categorical_cols:
    col_clean = col.strip()
    matching = [c for c in df_raw.columns if c.strip() == col_clean]
    if matching:
        vals = df_raw[matching[0]].value_counts(dropna=False)
        print(f'\n--- {col_clean} ({len(vals)} unique values) ---')
        print(vals.head(15).to_string())

In [ ]:
import datetime

metadata = {
    'Dataset Name': 'investments_VC.csv',
    'Source': 'Crunchbase via Kaggle (raw startup investment records)',
    'Extraction Date': str(datetime.date.today()),
    'Total Rows': df_raw.shape[0],
    'Total Columns': df_raw.shape[1],
    'File Size (MB)': round(os.path.getsize(RAW_DATA_PATH) / (1024 * 1024), 2),
    'Date Range (Founded)': f"{df_raw['founded_year'].min()} – {df_raw['founded_year'].max()}",
    'Countries': df_raw['country_code'].nunique(),
    'Startup Statuses': df_raw['status'].dropna().unique().tolist(),
}

print('=== EXTRACTION METADATA ===')
for k, v in metadata.items():
    print(f'{k:<30}: {v}')

## Takeouts

| Issue Observed | Column | Action Planned |
|---|---|---|
| Whitespace in column names | `market`, `funding_total_usd` | Strip in cleaning |
| Funding formatted as string with commas | `funding_total_usd` | Convert to numeric |
| Mixed date formats | `founded_at`, `first_funding_at`, `last_funding_at` | Parse to datetime |
| High missing rate | `state_code`, `region`, `city` | Impute or drop |
| Pipe-separated values | `category_list` | Parse in feature engineering |
| Startup name anomalies | `name` | Keep as-is, flag duplicates |